# Policy documents: chunk every `.docx` into `policy_chunks`

Reads **every `.docx` in `Files/Policies`** and splits it by heading into chunks. It does **not** look for any register tables and it does **not** map anything: a document is chunked whether or not it has an *In-scope data assets* table or a *Data Element Register*.

**What it writes**: one table, `policy_chunks` (shared by all documents, merged by `doc_id`). Only the documents found in this run are replaced; chunks of other documents are kept.

**Mapping is no longer done here.** Linking a policy chunk to the tables and columns it governs is done by the LLM mapping in the Ontology_Lineage app (Run jobs > Policy mapping): you review a dry run first, and nothing is written to Neo4j until you apply it. The old register-based mapping (`policy_loan_assets`, `policy_loan_data_elements`, `policy_loan_mapping_report`, links tagged `policy_register_notebook`) has been removed from this notebook.

**Then run `03_policy_chunks_to_neo4j_vector`** (loads all chunks and the vector index), and `04_embed_lineage_nodes_to_neo4j` if lineage changed.


In [ ]:
# Packages are installed from plain Python, not with the pip magic command: Fabric disables that magic in runs that
# are not interactive (a pipeline, a schedule, or the Run jobs button), which fails the whole session. Interactive runs work too.
import importlib, importlib.util, site, subprocess, sys
NEEDED = {"docx": "python-docx"}          # import name -> pip name
missing = [pip for mod, pip in NEEDED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    site.addsitedir(site.getusersitepackages())
    importlib.invalidate_caches()
for mod in NEEDED:
    importlib.import_module(mod)
print("packages ready:", sorted(NEEDED.values()))


## Part A. 1. Config

In [ ]:
import os

LAKE_ROOT  = "/lakehouse/default"
POLICY_DIR = f"{LAKE_ROOT}/Files/Policies"
INCLUDE_FILES = None      # None = every .docx under Files/Policies; or a list of file names

CHUNKS_PATH   = "Tables/dbo/policy_chunks"               # shared with the other policies (merged by doc_id)

MAX_SPLIT_LEVEL = 3
MAX_CHARS       = 3000

# doc_ids of old names of renamed documents, to purge from policy_chunks (doc_id = file name, lower case, dashes)
REMOVE_DOC_IDS = ["demobank-loan-policy",
                  "demobank-credit-risk-classification-policy",
                  "demobank-customer-data-protection-policy"]

## Part A. 2. Parser and chunker

In [ ]:
import re, glob, hashlib
import pandas as pd
from datetime import datetime, timezone
from docx import Document
from docx.table import Table
from docx.text.paragraph import Paragraph

def heading_level(p):
    name = (p.style.name or "") if p.style is not None else ""
    m = re.match(r"Heading\s+(\d+)", name, re.I)
    if m:
        return int(m.group(1))
    if name.strip().lower() == "title":
        return 0
    return None

_NUM_HEAD = re.compile(r"^(\d+(?:\.\d+){0,3})[.)]?\s+\S")
def numbered_level(p):
    t = p.text.strip()
    if not t or len(t) > 120:
        return None
    m = _NUM_HEAD.match(t)
    if not m:
        return None
    runs = [r for r in p.runs if r.text.strip()]
    if not runs or not all(r.bold for r in runs):
        return None
    return m.group(1).count(".") + 1

def iter_blocks(doc):
    for child in doc.element.body.iterchildren():
        tag = child.tag.rsplit("}", 1)[-1]
        if tag == "p":
            yield Paragraph(child, doc)
        elif tag == "tbl":
            yield Table(child, doc)

def table_text(tbl):
    rows = []
    for row in tbl.rows:
        cells = []
        for c in row.cells:
            t = " ".join(c.text.split())
            if not cells or t != cells[-1]:
                cells.append(t)
        line = " | ".join(x for x in cells if x)
        if line:
            rows.append(line)
    return "\n".join(rows)

def parse_docx(path):
    doc = Document(path)
    blocks = list(iter_blocks(doc))
    paras = [b for b in blocks if isinstance(b, Paragraph)]
    use_styles = any((heading_level(p) or 0) >= 1 for p in paras)
    level_fn = heading_level if use_styles else numbered_level
    title, sections, stack = None, [], []
    cur = {"heading": "(Preamble)", "level": 0, "path": "(Preamble)", "parts": []}
    for b in blocks:
        if isinstance(b, Table):
            t = table_text(b)
            if t:
                cur["parts"].append(t)
            continue
        text = b.text.strip()
        if not text:
            continue
        lvl = level_fn(b)
        if lvl == 0:
            title = title or text
            continue
        if lvl is not None and lvl <= MAX_SPLIT_LEVEL:
            if cur["parts"]:
                sections.append(cur)
            while stack and stack[-1][0] >= lvl:
                stack.pop()
            stack.append((lvl, text))
            cur = {"heading": text, "level": lvl, "path": " > ".join(t for _, t in stack), "parts": []}
        else:
            cur["parts"].append(text)
    if cur["parts"]:
        sections.append(cur)
    props = doc.core_properties
    meta = {"doc_title": (props.title or "").strip() or title or None,
            "author": (props.author or "").strip() or None,
            "doc_modified": props.modified.isoformat() if props.modified else None,
            "used_heading_styles": use_styles}
    return sections, meta

def split_long(text, max_chars):
    if len(text) <= max_chars:
        return [text]
    out, buf = [], ""
    for para in text.split("\n"):
        if buf and len(buf) + len(para) + 1 > max_chars:
            out.append(buf)
            buf = para
        else:
            buf = f"{buf}\n{para}" if buf else para
    if buf:
        out.append(buf)
    return out

def slug(s):
    return re.sub(r"[^A-Za-z0-9]+", "-", s).strip("-").lower()


## Part A. 3. Chunk every policy document

In [ ]:
files = sorted(glob.glob(os.path.join(POLICY_DIR, "**", "*.docx"), recursive=True))
files = [f for f in files if not os.path.basename(f).startswith("~$")]
if INCLUDE_FILES:
    files = [f for f in files if os.path.basename(f) in INCLUDE_FILES]
if not files:
    raise FileNotFoundError(f"No .docx files under {POLICY_DIR}. Upload the policies to Files/Policies (case-sensitive) "
                            "and make sure Ontology_Lineage is the default lakehouse.")

ingested_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
chunk_rows, processed = [], []

for path in files:
    fname = os.path.basename(path)
    doc_id = slug(os.path.splitext(fname)[0])
    sections, meta = parse_docx(path)
    if not meta["used_heading_styles"]:
        print(f"NOTE: {fname} has no Word Heading styles; used the numbered-heading fallback")
    with open(path, "rb") as fh:
        file_sha256 = hashlib.sha256(fh.read()).hexdigest()
    n_before = len(chunk_rows)
    for s_idx, s in enumerate(sections, start=1):
        for p_idx, piece in enumerate(split_long("\n".join(s["parts"]), MAX_CHARS), start=1):
            text = f"{s['path']}\n{piece}"
            chunk_rows.append({"chunk_id": f"{doc_id}-{s_idx:03d}-{p_idx:02d}", "doc_id": doc_id,
                               "doc_title": meta["doc_title"] or os.path.splitext(fname)[0], "source_file": fname,
                               "section_no": s_idx, "part_no": p_idx, "heading": s["heading"], "heading_level": s["level"],
                               "section_path": s["path"], "chunk_text": text, "char_count": len(text), "word_count": len(text.split()),
                               "author": meta["author"], "doc_modified": meta["doc_modified"], "file_sha256": file_sha256,
                               "ingested_at": ingested_at})
    processed.append({"doc_id": doc_id, "file": fname, "chunks": len(chunk_rows) - n_before})

dup_ids = pd.Series([p["doc_id"] for p in processed]).duplicated()
if dup_ids.any():
    raise ValueError("Two files produce the same doc_id (file names differ only in punctuation or case). Rename one.")
empty = [p["file"] for p in processed if p["chunks"] == 0]
if empty:
    print("WARNING: no text found (no chunks) in:", empty)
chunks_pdf = pd.DataFrame(chunk_rows)
if chunks_pdf.empty:
    raise ValueError("No chunks produced from any document.")
print(pd.DataFrame(processed).to_string(index=False))


## Part A. 4. Write to the lakehouse (these documents' rows are replaced, other documents are kept)

In [ ]:
CHUNK_DDL = ("chunk_id string, doc_id string, doc_title string, source_file string, section_no int, part_no int, "
             "heading string, heading_level int, section_path string, chunk_text string, char_count int, "
             "word_count int, author string, doc_modified string, file_sha256 string, ingested_at string")

def merge_delta(path, new_pdf, ddl, replace_ids, drop_ids=()):
    try:
        old = spark.read.format("delta").load(path).toPandas()
        old = old[~old["doc_id"].isin(set(replace_ids) | set(drop_ids))]
    except Exception:
        old = new_pdf.iloc[0:0]
    out = pd.concat([old, new_pdf], ignore_index=True)[list(new_pdf.columns)]
    out = out.astype(object).where(out.notna(), None)
    (spark.createDataFrame(out, schema=ddl).write.format("delta")
          .mode("overwrite").option("overwriteSchema", "true").save(path))
    return len(out), len(new_pdf)

replace_ids = [p["doc_id"] for p in processed]
total, mine = merge_delta(CHUNKS_PATH, chunks_pdf, CHUNK_DDL, replace_ids, REMOVE_DOC_IDS)
print(f"{CHUNKS_PATH}: {mine} rows written for {len(replace_ids)} document(s), {total} rows in total")


## Now run these notebooks (in this order), then map with the LLM
1. `03_policy_chunks_to_neo4j_vector`: loads all chunks (every document) into Neo4j and rebuilds the policy vector index.
2. `04_embed_lineage_nodes_to_neo4j`: only if the lineage nodes changed.
3. **Policy mapping (LLM)**, in the Ontology_Lineage app (Run jobs > Policy mapping). Run a dry run, review the proposed links, then apply.

Step 5 (`05_policy_map_to_graph_step5`) is retired: the LLM mapping replaces it.
